<a href="https://colab.research.google.com/github/Marconi-Lab/Swahili_ASR_Model/blob/main/Fine_tuning_(a_pre-trained_model)_for_Swahili_ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we aim to take a pre-trained model from hugging face, specifically, we are looking at;

 1.  [Wav2Vec2](https://ai.facebook.com/blog/wav2vec-20-learning-the-structure-of-speech-from-raw-audio/).
 
 2. [XLSR-Wav2Vec2](https://ai.facebook.com/blog/-xlm-r-state-of-the-art-cross-lingual-understanding-through-self-supervision/).
 there are three of these models:
[Wav2Vec2-XLS-R-300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m)
, [Wav2Vec2-XLS-R-1B](https://huggingface.co/facebook/wav2vec2-xls-r-1b)
and [Wav2Vec2-XLS-R-2B](https://huggingface.co/facebook/wav2vec2-xls-r-2b). We experiment on one of them at a time.
 
 3. [Whisper](https://huggingface.co/openai/whisper-large-v2). The most recent pre-trained model for ASR. Accompanied by an [article](https://arxiv.org/pdf/2212.04356.pdf) published in december 2022
 
 
 
 and fine-tuning with [swahili data](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0) from mozilla common voice hosted in hugging face dataset platform. 


## Install all the requirements

In [2]:
!nvidia-smi
!pip install datasets
!pip install transformers==4.11.3
!pip install torchaudio==0.10.0+cu113 -f https://download.pytorch.org/whl/cu113/torch_stable.html #Install version 0.10.0 with CUDA support for NVIDIA GPUs.
!pip install jiwer

Wed Feb  8 12:59:36 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 510.47.03    Driver Version: 510.47.03    CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   48C    P0    27W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

 We will use notebook_login() function to access token which we then use to authenticate to the Hugging Face Hub and allow us to download datasets,  models, and save our checkpoints during training. The Git Large File Storage (LFS) package will help us upload your model checkpoints:

In [3]:
from huggingface_hub import notebook_login
notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [4]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree       
Reading state information... Done
git-lfs is already the newest version (2.9.2-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-510
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.


## Data

In this stage, we download the common voice data, and the prepare it for fine-tuning one of the three pre-trained models we mentioned at the beginning.

In [5]:
from datasets import load_dataset

training_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="train")
testing_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="test")

Computing checksums:   8%|8         | 1/12 [00:05<01:02,  5.66s/it]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 7174it [00:00, 71618.83it/s]
Reading metadata...: 14336it [00:00, 68858.38it/s]
Reading metadata...: 26614it [00:00, 62555.14it/s]


...

Generating other split: 0 examples [00:00, ? examples/s]





Reading metadata...: 0it [00:00, ?it/s]



Reading metadata...: 10694it [00:00, 106933.47it/s]



Reading metadata...: 25226it [00:00, 129508.93it/s]



Reading metadata...: 38533it [00:00, 131133.00it/s]



Reading metadata...: 52144it [00:00, 133094.35it/s]



Reading metadata...: 65899it [00:00, 134699.11it/s]



Reading metadata...: 79369it [00:00, 129382.69it/s]



Reading metadata...: 92346it [00:00, 123753.57it/s]



Reading metadata...: 104782it [00:00, 116560.79it/s]



Reading metadata...: 116716it [00:00, 117362.61it/s]



Reading metadata...: 128914it [00:01, 118709.47it/s]



Reading metadata...: 140843it [00:01, 118673.56it/s]



Reading metadata...: 152750it [00:01, 102158.38it/s]



Reading metadata...: 163372it [00:01, 92801.57it/s] 



Reading metadata...: 173039it [00:01, 87471.92it/s]



Reading metadata...: 182062it [00:01, 80087.42it/s]



Reading metadata...: 190311it [00:01, 78089.45it/s]



Reading metadata...: 198266it [00:01, 77674.78it/s]



Reading meta

Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 9599it [00:00, 95981.85it/s]
Reading metadata...: 22847it [00:00, 117356.85it/s]
Reading metadata...: 47470it [00:00, 122066.18it/s]


Dataset common_voice_11_0 downloaded and prepared to /root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/sw/11.0.0/2c65b95d99ca879b1b1074ea197b65e0497848fd697fdb0582e0f6b75b6f4da0. Subsequent calls will reuse this data.


In [6]:
training_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 26614
})

In [8]:
testing_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 10238
})

###  We observe that:

For training data, we have 11 columns : `['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment']`, and `26614` rows. 

For testing data, we have the same number of columns but now `10238` rows.
    

###  Let try exploring the data:

1. Remove the unrequired columns from both training and testing set
2. Then output 10 random sentences from the trainin set

In [10]:
#we only remain with the path, audio and sentence which are the only columns the model will require for training
training_data = training_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
testing_data = testing_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

In [12]:
# we only have the path, audio and sentence as we expected
print(training_data)
print(testing_data)

Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 26614
})
Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 10238
})


In [9]:
#we now generate 10 random sentences from our two datasets
import random
import pandas as pd
from IPython.display import display, HTML
from datasets import ClassLabel

#this function will receive a dataset then output 10 random sentences

def display_random_elements(dataset, num_examples=10):

  #we first confirm that the dataset is more than 10 sentences
    if num_examples > len(dataset):
        raise ValueError("Can't pick more elements than there are in the dataset.")

  # returns a list of unique, randomly selected integers from 0 to dataset-1
    random_indices = random.sample(range(len(dataset)), num_examples)

    # converts the picked examples into a Pandas DataFrame and displays
    df = pd.DataFrame(dataset[random_indices])
    display(HTML(df.to_html()))

In [1]:
# lets see any 10 examples on the training set
display_random_elements(training_data.remove_columns(["path", "audio"]), num_examples=10)

NameError: ignored

In [15]:
#lets also see 10 from testing set
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Mthezaji mzuri yule
1,"Data, kibinafsi haina maana yoyote."
2,Iliyokuwa ofisi ya mkuu wa wilaya posta ya zamani
3,"Hapo awali alikuwa Afisa Mkuu Mtendaji wa Taasisi ya Utawala bora, Kenya, mjini Nairobi."
4,Shughuli hii sasa imeiva
5,kwa kuwatangazia watu zawadi za mamilioni ya dola
6,Baba nipe pesa
7,limekuwa na msururu wa mikutano mapema hii leo
8,Cheka uongeze maisha
9,Wachaga hutania kwa kusema kuwa wapare wanakula ugali kwa picha ya samaki


Let's extract all distinct letters of the training and test data and build our vocabulary from this set of letters.